# An introduction to temporal point processes

A temporal point process (TPP) models events in continuous time, each with a type. This notebook
builds a small Hawkes process with hand-picked parameters and computes, step by step, every quantity
a TPP is made of: the conditional intensity, the cumulative intensity, the survival function, the
next-event densities, the log-likelihood and sampling. Each step uses the public TorchPP call and
checks it against the formula.

The same material, without code, is in [`docs/introduction.md`](../docs/introduction.md). Runs on CPU
in a few seconds.

In [ ]:
import torch
import matplotlib.pyplot as plt
import torchpp as tpp
from torchpp.modules.intensity import ConstantBaselineParams, ExpKernelParams

torch.set_default_dtype(torch.float64)  # exact comparisons below
COLORS = ["tab:blue", "tab:red"]

## A Hawkes process with two event types

A Hawkes process has a baseline rate $\mu_d$ per type and a kernel that adds a bump after every event.
With the exponential kernel, an event of type $j$ at time $t_i$ raises the rate of type $d$ by

$$\phi_{d,j}(t - t_i) = \alpha_{dj}\, \beta_{dj}\, e^{-\beta_{dj}(t - t_i)}.$$

- $\alpha_{dj}$: expected number of type-$d$ events that one type-$j$ event triggers.
- $\beta_{dj}$: how fast the effect fades (1 / mean delay).

We pick the parameters by hand. Type 0 excites itself and type 1; type 1 only excites itself.
Matrices are indexed `[target d, source j]`. `token_roles=TokenRoles(has_origin=True)` means time 0 is a real start
(nothing before it), so we can simulate from an empty history.

In [ ]:
mu = torch.tensor([0.3, 0.2])
alpha = torch.tensor([[0.4, 0.0],    # type 0 is excited by type 0
                      [0.3, 0.5]])   # type 1 is excited by type 0 and by type 1
beta = torch.tensor([[2.0, 1.0],
                     [1.0, 1.5]])


def make_hawkes(mu, alpha, beta):
    return tpp.ConstBaselineExpKernelHawkesProcess(
        D=2,
        baseline_params=ConstantBaselineParams(mu=mu),
        kernel_params=ExpKernelParams(alpha=alpha, beta=beta),
        token_roles=tpp.TokenRoles(has_origin=True),
    )


model = make_hawkes(mu, alpha, beta)
with torch.no_grad():
    print("branching matrix (= alpha):\n", model.branching_matrix())

## Simulate a sequence

`model.sample(history, num_steps)` draws events one at a time. Starting from no history:

In [ ]:
with torch.no_grad():
    sim = model.sample(None, num_steps=25, rng=1).sequence
T_obs = float(sim.time_points[-1]) + 1.0          # we kept observing 1 time unit after the last event
seq = tpp.MVEventData(sim.time_points, sim.event_types, T_obs=T_obs)
batch = tpp.BatchedMVEventData(mv_events=[seq])
print(f"{len(seq)} events on [0, {T_obs:.2f}]")
print("times:", [round(x, 2) for x in seq.time_points.tolist()[:8]], "...")
print("types:", seq.event_types.tolist()[:8], "...")

## The conditional intensity

$\lambda_d(t)$ is the rate of type-$d$ events at time $t$, given everything before $t$:
$\lambda_d(t)\,\Delta t \approx \Pr(\text{type-}d \text{ event in } [t, t+\Delta t) \mid \text{history})$.
`model.intensity(t, batch)` evaluates it; with query times of shape `(B, Q)` it returns `(B, Q, D)`.
The rate jumps at every event and decays back to the baseline: events cluster.

In [ ]:
grid = torch.linspace(0, T_obs, 3000)
with torch.no_grad():
    lam = model.intensity(grid.unsqueeze(0), batch)[0]          # (Q, D)

fig, ax = plt.subplots(figsize=(10, 3.2))
for d in range(2):
    ax.plot(grid, lam[:, d], color=COLORS[d], label=f"$\\lambda_{d}(t)$")
    ev = seq.time_points[seq.event_types == d]
    ax.plot(ev, torch.full_like(ev, -0.15), "|", color=COLORS[d], markersize=14)
ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("time"); ax.set_ylabel("intensity"); ax.legend(); ax.set_title("Conditional intensity; ticks = events")
plt.show()

## From the intensity to the next event

Stop the history after 8 events and ask: when does the next event happen, and of which type?
Call $t_0$ the time of the last event in the history.

- **Cumulative intensity** $\Lambda(t_0, t) = \sum_d \int_{t_0}^t \lambda_d(s)\,ds$:
  `cumulative_intensity(t, batch, T_low=t0)`.
- **Survival** $S(t) = \exp(-\Lambda(t_0, t))$, the probability of no event in $(t_0, t]$: `survival_time`.
- **Density of the next time** $p(t) = \lambda(t)\,S(t)$ with $\lambda = \sum_d \lambda_d$: `density_time`.
- **Type given the time** $p(d \mid t) = \lambda_d(t)/\lambda(t)$: `prob_mark_given_time`.

All of them return logarithms by default; `log=False` gives the plain values.

In [ ]:
hist = tpp.MVEventData(seq.time_points[:8], seq.event_types[:8])
hist_b = tpp.BatchedMVEventData(mv_events=[hist])
t0 = float(hist.time_points[-1])
q = torch.linspace(t0 + 1e-4, t0 + 8, 400).unsqueeze(0)         # (1, Q) query times, just after t0
# (exactly at t0 the intensity uses the events strictly before t0, i.e. not the last event's jump)

with torch.no_grad():
    lam_q = model.intensity(q, hist_b)[0]                        # (Q, D)
    S = model.survival_time(q, hist, log=False)[0]               # (Q,)
    p_t = model.density_time(q, hist, log=False)[0]
    p_d_given_t = model.prob_mark_given_time(q, hist, log=False)[0]   # (Q, D)
    Lam = model.cumulative_intensity(torch.tensor([t0 + 2.0]), hist_b, T_low=torch.tensor([t0])).sum()
    S2 = model.survival_time(torch.tensor([t0 + 2.0]), hist, log=False)

print(f"S(t0+2) = {S2.item():.6f}   exp(-Lambda(t0, t0+2)) = {torch.exp(-Lam).item():.6f}")
print(f"p(t) = lambda(t) S(t): max difference {(p_t - lam_q.sum(-1) * S).abs().max().item():.2e}")

x = q[0] - t0
fig, axs = plt.subplots(1, 3, figsize=(13, 3.2))
axs[0].plot(x, S, "k"); axs[0].set_title("survival $S(t)$")
axs[1].plot(x, p_t, "k"); axs[1].set_title("next-event density $p(t)$")
for d in range(2):
    axs[2].plot(x, p_d_given_t[:, d], color=COLORS[d], label=f"type {d}")
axs[2].set_title("type given the time $p(d \\mid t)$"); axs[2].legend()
for a in axs:
    a.set_xlabel("time since the last event")
plt.tight_layout(); plt.show()

Right after the last event the rate is high, so $S$ drops fast and $p(t)$ peaks early. Later the rate
decays to the baseline and $p(t)$ becomes an exponential tail.

Summaries of the next event: `time_mean`, `time_quantile(u, ...)` and `prob_mark` ($p(d)$, the type
whatever the time).

In [ ]:
with torch.no_grad():
    print(f"last event at t0 = {t0:.2f}")
    print("mean time of the next event:", round(model.time_mean(hist).item(), 3))
    print("median time of the next event:", round(model.time_quantile(0.5, hist).item(), 3))
    print("p(d) =", model.prob_mark(hist, log=False)[0].numpy().round(3))

## The log-likelihood

Chain the next-event densities, $p(t_i, m_i) = \lambda_{m_i}(t_i)\exp(-\Lambda(t_{i-1}, t_i))$, and add the
probability that nothing happened after the last event until $T_\mathrm{obs}$. The survival terms join
into one integral:

$$\log L = \sum_i \log \lambda_{m_i}(t_i) - \Lambda(0, T_\mathrm{obs}).$$

At an event time the intensity uses only the events strictly before it. We compute both terms with
`intensity` and `cumulative_intensity` and compare with `model.likelihood`.

In [ ]:
with torch.no_grad():
    lam_at_events = model.intensity(seq.time_points.unsqueeze(0), batch)[0]      # (N, D)
    log_rates = torch.log(lam_at_events[torch.arange(len(seq)), seq.event_types]).sum()
    total = model.cumulative_intensity(torch.tensor([T_obs]), batch, T_low=torch.tensor([0.0])).sum()
    by_hand = log_rates - total
    print(f"sum log lambda = {log_rates.item():.4f}   Lambda(0, T_obs) = {total.item():.4f}")
    print(f"by hand: {by_hand.item():.6f}   model.likelihood: {model.likelihood(batch).item():.6f}")

The first term rewards a high rate where events happened; the second charges the model for rate
everywhere, including the quiet stretch after the last event. Fitting a model maximises this number
(`model.loss` is its negative).

## Sampling by inversion

The integrated rate up to the next event, $\Lambda(t_0, t_\mathrm{next})$, follows an exponential
distribution with rate 1. So: draw $u \sim \mathrm{Exp}(1)$, solve $\Lambda(t_0, t) = u$ for $t$, then draw
the type from $p(d \mid t)$. That is what `sample` does by default. Many draws should follow $p(t)$:

In [ ]:
with torch.no_grad():
    draws = torch.stack([model.sample(hist, num_steps=1, rng=i).times[0] for i in range(1000)]).squeeze(-1) - t0

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.hist(draws.numpy(), bins=60, range=(0, 8), density=True, alpha=0.5, label="1000 samples")
ax.plot(x, p_t, "k", label="$p(t)$")
ax.set_xlabel("time since the last event"); ax.legend(); plt.show()

## Play with the parameters

Change $\mu$, $\alpha$ and $\beta$ (the same value for every pair of types here; $\alpha$ stays below
0.5 because each type is excited by both types, so $2\alpha < 1$ keeps the process stable) and watch the intensity and the
log-likelihood of the simulated sequence change. A larger $\alpha$ means more clustering; a larger $\beta$
means shorter bursts. The sliders need `pip install ipywidgets`; without it, the cell draws three fixed
settings.

In [ ]:
def draw(mu_=0.3, alpha_=0.2, beta_=1.5):
    m = make_hawkes(torch.full((2,), mu_), torch.full((2, 2), alpha_), torch.full((2, 2), beta_))
    with torch.no_grad():
        l = m.intensity(grid.unsqueeze(0), batch)[0].sum(-1)
        ll = m.likelihood(batch).item()
    plt.figure(figsize=(10, 2.6))
    plt.plot(grid, l, "k")
    plt.plot(seq.time_points, torch.zeros(len(seq)), "|", color="tab:gray", markersize=12)
    plt.title(f"ground intensity  (mu={mu_:.2f}, alpha={alpha_:.2f}, beta={beta_:.2f});  log L = {ll:.2f}")
    plt.xlabel("time"); plt.show()


try:
    import ipywidgets as widgets
    widgets.interact(draw, mu_=(0.05, 1.0, 0.05), alpha_=(0.0, 0.45, 0.05), beta_=(0.2, 5.0, 0.1))
except ImportError:
    for setting in [(0.3, 0.0, 1.5), (0.3, 0.2, 1.5), (0.1, 0.45, 3.0)]:
        draw(*setting)

With $\alpha = 0$ the model is a Poisson process: a flat rate that ignores the history.

## Next steps

- [`06_compare_models.ipynb`](./06_compare_models.ipynb): fit Poisson, Hawkes and neural TPPs to data
  simulated from this kind of model, and recover $\alpha$.
- [`01_quickstart.ipynb`](./01_quickstart.ipynb): the data structures and the full model interface.
- [`docs/introduction.md`](../docs/introduction.md) and [`docs/notation.md`](../docs/notation.md): the
  derivations and the reference.